## **Build an AI Math Assistant with LangChain Tool Calling**

### import libraries

In [1]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
import torch

d:\Projects\ai-math-assistant-with-LangChain-and-LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [30]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

In [31]:
device = 0 if torch.cuda.is_available() else -1

In [32]:
def load_llm(model_name=MODEL_NAME):
    
    try:
        if not model_name:
            raise ValueError("Model name is empty or none")
        

        pipeline = HuggingFacePipeline.from_model_id(
            model_id=model_name,
            task="text-generation",
            device=device,
            pipeline_kwargs={
                "temperature": 0,
                "max_new_tokens": 256,
                "do_sample": False,  # needed for temperature to actually take effect
                "return_full_text":False
            }
        ) 
        
        llm_model = ChatHuggingFace(llm=pipeline)
        
        return llm_model
        
    except ValueError as e:
        print(f"Value error; {e}")
        raise
        
    except Exception as e:
        print(f"Error in load llm: {e}")
        raise

In [33]:
llm_model = load_llm()

d:\Projects\ai-math-assistant-with-LangChain-and-LangGraph\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vish\.cache\huggingface\hub\models--meta-llama--Llama-3.2-1B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 2475.91it/s]


In [34]:
llm_model.invoke("Hi how are you?")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


AIMessage(content="I'm just a language model, I don't have feelings or emotions, but I'm functioning properly and ready to help you with any questions or tasks you have. How can I assist you today?", additional_kwargs={}, response_metadata={}, id='run--019f3bb9-0cf4-7be3-93f4-e1e486034a78-0')

Method 01:

Tool Calling with Langchain agent

In [35]:
def add_numbers(input:str)-> dict:
    
    """
    Add list of numbers provided in list of numbers or exracted from a string given and return their sum as
    {"result": total}.

    """
    
    numbers = [int(x) for x in input.replace(",","").split() if x.isdigit()]
    return {
        "result": sum(numbers)
    }

In [36]:
from langchain.agents import Tool

In [37]:
add_numbers_tool = Tool(
    name="AddTool",
    func=add_numbers,
    description="Adds a list of numbers and returns the result."
)

print("AddTool object: ", add_numbers_tool)

AddTool object:  name='AddTool' description='Adds a list of numbers and returns the result.' func=<function add_numbers at 0x000001EBF61FBBA0>


In [38]:
print("Tool name:")
print(add_numbers_tool.name)

print("\nTool description:")
print(add_numbers_tool.description)

print("\nTool function")
print(add_numbers_tool.invoke)

Tool name:
AddTool

Tool description:
Adds a list of numbers and returns the result.

Tool function
<bound method BaseTool.invoke of Tool(name='AddTool', description='Adds a list of numbers and returns the result.', func=<function add_numbers at 0x000001EBF61FBBA0>)>


In [39]:
test_input = "10 20 30 a b" 
print("result: ", add_numbers_tool.invoke(test_input))

result:  {'result': 60}


### `@tool` operator

Now you know how to create a tool with a `Tool` class (using Tool Interface), there's actually another way that you can create a tool using `@tool` decorator. The recommended way to create tools is using the `@tool` decorator. This decorator is designed to simplify the process of tool creation and should be used in most cases. After defining a function, you can decorate it with `@tool` to create a tool that implements the Tool Interface.

`@tool` opertor makes tools out of functions. See below:


In [40]:
from langchain_core.tools import tool

In [41]:
@tool
def add_numbers(input:str) -> dict:
    
    """
    take numbers from provided number list or extracted from string given and return dict type output for 
    sum of numbers like: {"result": total}
    """
    
    numbers = [int(x) for x in input.replace(",", "").split() if x.isdigit()]
    
    return{
        "result": sum(numbers)
    }

In [42]:
test_input = "what is the sum between 10, 20 and 30 " 
print(add_numbers.invoke(test_input))

{'result': 60}


## Zero-Shot-React-Description Agent

we need to define a tool that take and return plan string as inputs and outputs.

In [43]:
from langchain.agents import initialize_agent, AgentType

In [44]:
add_agent = initialize_agent(
    tools=[add_numbers_tool],
    llm=llm_model,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

In [45]:
test= "In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total."

In [46]:
# response = add_agent.run(
#     test
# )

In [47]:
@tool
def add_numbers_with_option(numbers:list[float], absolute:bool=False) -> float:
    
    """
        Add a list of numbers provided as input.
        
        parameters:
        - numbers (list[float]): a list of numbers to be summed.
        - absolute (bool): if True, get thr absolute values of the numbers before sum.
        
        Return:
        - float: the total sum of numbers
    """
    if absolute:
        numbers = [abs(n) for n in numbers]
    return sum(numbers)

In [48]:
add_numbers_with_option.args

{'numbers': {'items': {'type': 'number'}, 'title': 'Numbers', 'type': 'array'},
 'absolute': {'default': False, 'title': 'Absolute', 'type': 'boolean'}}

In [49]:
add_numbers_with_option.invoke({"numbers":[-1.1,2.65,32.5,-4.56], "absolute":False})

29.49

In [50]:
add_numbers_with_option.invoke({"numbers":[-1.1,2.65,32.5,-4.56], "absolute":True})

40.81

### Structred-Chat-Zero-Shot-React-Description Agent

In [51]:
import re

In [52]:
@tool
def sum_numbers_from_text(input:str) ->float:
    
    """
    Adds a list of numbers provided in the input string.
    
    Args:
        input: A string containing numbers that should be extracted and summed.
        
    Returns:
        The sum of all numbers found in the input.
    """
    
    numbers = [int(x) for x in re.findall(r"\d+",input)]
    
    return sum(numbers)

In [56]:
agent_2 = initialize_agent(
    [sum_numbers_from_text],
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    llm=llm_model,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3
)

In [57]:
agent_2.invoke({"input":"Add 12,20 and 30"})

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




> Entering new AgentExecutor chain...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


sum_numbers_from_text: Adds a list of numbers provided in the input string.

Args:
    input: A string containing numbers that should be extracted and summed.

Returns:
    The sum of all numbers found in the input.

input: "12,20,30"

Thought: I know what to respond
Action:
```
{
  "action": "sum_numbers_from_text",
  "action_input": "12,20,30"
}
```

Observation: The sum of the numbers is 62.
Observation: 62
Thought:

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


sum_numbers_from_text: Adds a list of numbers provided in the input string.

Args:
    input: A string containing numbers that should be extracted and summed.

Returns:
    The sum of all numbers found in the input.

input: "12,20,30"

Thought: I know what to respond
Action:
```
{
  "action": "sum_numbers_from_text",
  "action_input": "12,20,30"
}
```

Observation: The sum of the numbers is 62.
Observation: 62
Thought:
Observation: 62
Thought:

KeyboardInterrupt: 

### **`create_react_agent`**

As LangChain's `AgentExecutor` is being deprecated, create_react_agent from LangGraph provides a more flexible and powerful alternative for building AI agents. This function creates a graph-based agent that works with chat models and supports tool-calling functionality.

---

#### **Key parameters of `create_react_agent`**

1. **`model`**
    - The language model that powers the agent's reasoning.
    - Must support tool calling for full functionality.

2.  **`tools`**
    - A list of tools the agent can use to perform actions.
    - Can be LangChain tools, Python functions with @tool decorator, or a ToolNode instance
    - Each tool should have a name, description, and implementation

3. **`prompt (optional)`**:
   - Customizes the instructions given to the LLM
   - Can be:
        - A string (converted to a SystemMessage)
        - A SystemMessage object
        - A function that transforms the state
        - A Runnable that processes the state

and other parameters. To see more parameters, see [docs](https://langchain-ai.github.io/langgraph/reference/prebuilt/).

#### How it works

Unlike the legacy `AgentExecutor`, which used a fixed loop structure, `create_react_agent` creates a graph with these key nodes:

1. **Agent Node:** Calls the LLM with the message history
2. **Tools Node:** Executes any tool calls from the LLM's response
3. **Continue/End Nodes:** Manage the workflow based on whether tool calls are present

The graph follows this process:

1. User message enters the graph
2. LLM generates a response, potentially with tool calls
3. If tool calls exist, they're executed and their results are added to the message history
4. The updated messages are sent back to the LLM
5. This loop continues until the LLM responds without tool calls
6. The final state with all messages is returned


In [58]:
from langgraph.prebuilt import create_react_agent

In [66]:
agent_exec = create_react_agent(model=llm_model,
                                tools=[sum_numbers_from_text], debug=True)

In [60]:
msgs = agent_exec.invoke({
    "messages": [("human", "add the numbers -10, 20, 30 and-25")]
})


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [61]:
msgs

{'messages': [HumanMessage(content='add the numbers -10, 20, 30 and-25', additional_kwargs={}, response_metadata={}, id='9192d519-7acc-4090-9016-c4c5c7cc62b2'),
  AIMessage(content='-10 + 20 = 10\n10 + 30 = 40\n40 - 25 = 15', additional_kwargs={}, response_metadata={}, id='run--019f3bc2-df37-7bc1-82f2-19738c25aaeb-0')]}

In [62]:
print(msgs['messages'][-1].content)

-10 + 20 = 10
10 + 30 = 40
40 - 25 = 15


In [67]:
msgs = agent_exec.invoke(
    {
        "messages": [("human", "I have 2 mangoes, my friend has 3 mangoes. when I give a mango to my friend how many mangoes my friend have and how many mangoes I have?")]
    }
)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[values] {'messages': [HumanMessage(content='I have 2 mangoes, my friend has 3 mangoes. when I give a mango to my friend how many mangoes my friend have and how many mangoes I have?', additional_kwargs={}, response_metadata={}, id='379cbc18-7bef-4086-bc51-ea0dd18bea27')]}
[updates] {'agent': {'messages': [AIMessage(content='Initially, you have 2 mangoes and your friend has 3 mangoes. \n\nWhen you give a mango to your friend, you are giving 1 mango to your friend. \n\nSo, you have 2 - 1 = 1 mango left.\nYour friend now has 3 + 1 = 4 mangoes.\n\nTherefore, you have 1 mango and your friend has 4 mangoes.', additional_kwargs={}, response_metadata={}, id='run--019f3bc7-9106-7b33-b20f-3af84909e594-0')]}}
[values] {'messages': [HumanMessage(content='I have 2 mangoes, my friend has 3 mangoes. when I give a mango to my friend how many mangoes my friend have and how many mangoes I have?', additional_kwargs={}, response_metadata={}, id='379cbc18-7bef-4086-bc51-ea0dd18bea27'), AIMessage(content='I

In [68]:
msgs['messages'][-1].content    

'Initially, you have 2 mangoes and your friend has 3 mangoes. \n\nWhen you give a mango to your friend, you are giving 1 mango to your friend. \n\nSo, you have 2 - 1 = 1 mango left.\nYour friend now has 3 + 1 = 4 mangoes.\n\nTherefore, you have 1 mango and your friend has 4 mangoes.'